# 01 — Verify & extract OSCD

Confirms the three OSCD zips are on Drive, extracts them to scratch, and runs the layout-discovery parser to summarize what it found.

**Storage contract** — code lives on GitHub, data & checkpoints live on Drive (`MyDrive/s2cd`), `/content` is disposable scratch. Every notebook is self-contained: run all cells top-to-bottom in a fresh GPU session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os

REPO_URL = "https://github.com/Aadith-Geeth-Mohan/sentinel2-change-detection.git
REPO = "/content/s2-change-detection"
if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone {REPO_URL} {REPO}
%cd {REPO}
!pip install -q -e .


In [ ]:
import torch
from s2cd.data import DriveStore
from s2cd.utils import get_device, describe_device, load_config

device = get_device()
print(describe_device(device))
store = DriveStore()  # Drive root: MyDrive/s2cd · scratch: /content


In [ ]:
zips = store.oscd_zips()  # raises with instructions if missing
for z in zips:
    print(f"{z.name}: {z.stat().st_size / 1e9:.2f} GB")


In [ ]:
oscd_root = store.extract_oscd()  # idempotent per session
print(oscd_root)


In [ ]:
from s2cd.data.oscd import list_cities, read_city

cities = list_cities(oscd_root)
print("train cities:", len(cities["train"]), sorted(cities["train"]))
print("test cities: ", len(cities["test"]), sorted(cities["test"]))

bands = load_config("configs/data.yaml")["bands"]
sample = sorted(cities["train"])[0]
city = read_city(oscd_root, sample, bands)
print(f"\nsample city {sample!r}: t0 {city['t0'].shape}, "
      f"dates {city['dates'][0]} -> {city['dates'][1]}")
print("change pixels:", int(city["mask"].sum()), "/", city["mask"].size)
